# الفصل 60: نماذج الاستدلال وTest-Time Compute

> طبقة الحدس حتى عام 2024، كانت الفلسفة السائدة لتحسين LLM هي: "درِّب نموذجًا أكبر على بيانات أكثر" (Scaling Laws الكلاسيكية).

- **الجزء:** 12 — الاتجاهات المستقبلية
- **الفصل:** 60
- **المستوى:** خبير
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 59: إدارة التكلفة ونضج LLMOps
- **الوسوم:** Reasoning، DeepSeek-R1، Test-Time Compute، PRM

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch60-reasoning-models.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس حتى عام 2024، كانت الفلسفة السائدة لتحسين LLM هي: "درِّب نموذجًا أكبر على بيانات أكثر" (Scaling Laws الكلاسيكية).

> **بيانات الفصل**
>
> - **الجزء:** 12 — الاتجاهات المستقبلية
> - **المستوى:** خبير
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 59: إدارة التكلفة ونضج LLMOps


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم من Chain-of-Thought إلى "التفكير الداخلي" المُدرَّب
- فهم آلية DeepSeek-R1 (الأكثر توثيقًا علنيًا)
- فهم ظاهرة "لحظة الأها" (Aha Moment) والـ Backtracking


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**
> حتى عام 2024، كانت الفلسفة السائدة لتحسين LLM هي: "درِّب نموذجًا أكبر على بيانات أكثر" (Scaling Laws الكلاسيكية). لكن DeepSeek-R1 وOpenAI o1/o3 أثبتت وجود **محور تحجيم ثانٍ** مستقل تمامًا: بدلًا من زيادة حجم النموذج، اترك النموذج **"يفكر" لفترة أطول وقت الاستدلال** (Test-Time Compute) قبل أن يُخرج الإجابة النهائية. هذا غيّر قواعد اللعبة: نموذج صغير نسبيًا، إذا "فكّر" لمدة أطول (آلاف الرموز من التفكير الداخلي)، قد يتفوق على نموذج أضخم منه بعشرة أضعاف لكنه يُجيب مباشرة.


## 2. الخريطة الذهنية


```text
محوران للتحجيم (Scaling)
│
├── Train-Time Scaling (التقليدي)
│   N (معاملات) ↑ + D (بيانات) ↑ → أداء أفضل
│   (Chinchilla Laws - الفصل 20)
│
└── Test-Time Scaling (الجديد - 2024+)
    وقت التفكير/عدد الرموز المُستهلكة وقت الاستدلال ↑ → أداء أفضل
    (o1, o3, DeepSeek-R1, QwQ)
```


## 3. الشرح المتدرج

### 3.1 من Chain-of-Thought إلى "التفكير الداخلي" المُدرَّب

في الفصل 52 تعرّفنا على **Chain-of-Thought (CoT) Prompting**: مجرد طلب "فكّر خطوة بخطوة" من نموذج مُدرَّب عاديًا. لكن نماذج الاستدلال (Reasoning Models) مثل o1 وDeepSeek-R1 **مُدرَّبة خصيصًا** — عبر RL — على توليد سلاسل تفكير طويلة جدًا (قد تصل لآلاف الرموز) **قبل** الإجابة النهائية، وهذه السلاسل غير مرئية للمستخدم في حالة o1 (تُسمى "hidden reasoning tokens").

> 🟡 **طبقة الآلية: الفرق الجوهري**

| الخاصية | CoT Prompting العادي | Reasoning Models (o1/R1) |
|---|---|---|
| مصدر القدرة | Prompt فقط على نموذج عادي | تدريب RL مخصص على التفكير |
| طول التفكير | محدود (بضع مئات من الرموز) | ديناميكي، قد يصل لآلاف الرموز |
| القدرة على التراجع | محدودة جدًا | **Backtracking** مُدرَّب (يكتشف خطأه ويعيد المحاولة) |
| التكلفة وقت الاستدلال | ثابتة تقريبًا | تتناسب طرديًا مع صعوبة السؤال |
| التحسّن مع مزيد من "الوقت" | لا يتحسّن كثيرًا بعد حد معين | يتحسّن بشكل شبه رتيب (log-linear) |

### 3.2 آلية DeepSeek-R1 (الأكثر توثيقًا علنيًا)

خط أنابيب DeepSeek-R1 يكشف المنهجية بوضوح نادر في هذا المجال:


```text
المرحلة 1: DeepSeek-R1-Zero
نموذج أساسي (DeepSeek-V3-Base)
    → تدريب RL مباشر (بدون SFT!) باستخدام GRPO
    → يتعلّم "التفكير الطويل" ذاتيًا من مكافأة الصحة فقط
    → مشكلة: لغة مختلطة، تفكير غير مقروء أحيانًا

المرحلة 2: DeepSeek-R1 (النسخة النهائية)
نموذج أساسي
    → "Cold Start" SFT على آلاف أمثلة تفكير عالية الجودة (يدويًا/مُولَّدة)
    → RL موجّه نحو الاستدلال (Reasoning-oriented RL) + مكافأة قابلية القراءة
    → Rejection Sampling لتوليد بيانات SFT جديدة من النموذج نفسه
    → SFT ثانٍ على مزيج (بيانات استدلال + بيانات عامة)
    → RL نهائي لكل السيناريوهات (Helpfulness + Harmlessness)
```


> 🔴 **طبقة التعمّق: خوارزمية GRPO (Group Relative Policy Optimization)**

GRPO هي البديل الذي استخدمته DeepSeek بدلًا من PPO، وهي أبسط حسابيًا لأنها **لا تحتاج نموذج Critic (Value Model) منفصل** — توفّر كبير في الذاكرة.

**آلية PPO التقليدية (تذكير من الفصل 31):**


```text
Policy Model يولّد استجابة → Reward Model يُقيّمها → Value Model يُقدّر baseline
→ Advantage = Reward - Value(state)
→ تحديث Policy بناءً على Advantage مع قص (clipping)
```


**آلية GRPO:**


```text
لكل سؤال q، وَلِّد مجموعة من G استجابات (مثلاً G=16) من نفس السياسة الحالية
احسب المكافأة r_i لكل استجابة i في المجموعة
Advantage النسبي = (r_i - mean(r_1...r_G)) / std(r_1...r_G)
→ لا حاجة لنموذج Value منفصل! المجموعة نفسها هي الـ baseline
```


**الكود الكامل لـ GRPO المبسّط:**


In [ ]:
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Callable

@dataclass
class GRPOConfig:
    group_size: int = 16          # عدد الاستجابات لكل سؤال
    clip_epsilon: float = 0.2     # قص PPO-style
    kl_coefficient: float = 0.04  # عقوبة الابتعاد عن النموذج المرجعي
    learning_rate: float = 1e-6

class GRPOTrainer:
    """
    تطبيق مبسّط ومفاهيمي لخوارزمية GRPO المستخدمة في تدريب DeepSeek-R1.
    الفكرة الجوهرية: استبدال Value Model بمتوسط مكافآت المجموعة.
    """
    def __init__(self, policy_model, ref_model, reward_fn: Callable,
                 config: GRPOConfig):
        self.policy_model = policy_model
        self.ref_model = ref_model  # نسخة مجمّدة من النموذج قبل RL (لـ KL)
        self.reward_fn = reward_fn  # دالة مكافأة قابلة للتحقق (مثل: صحة الإجابة الرياضية)
        self.config = config
        self.optimizer = torch.optim.AdamW(
            self.policy_model.parameters(), lr=config.learning_rate
        )

    def sample_group(self, prompt_ids, tokenizer, max_new_tokens=2048):
        """توليد G استجابات مختلفة لنفس السؤال (بعينات Sampling عشوائية)."""
        group_size = self.config.group_size
        responses = []
        for _ in range(group_size):
            with torch.no_grad():
                output_ids = self.policy_model.generate(
                    prompt_ids,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=1.0,
                    top_p=0.95,
                )
            responses.append(output_ids)
        return responses

    def compute_group_advantages(self, rewards: List[float]) -> torch.Tensor:
        """حساب الـ Advantage النسبي داخل المجموعة (بديل Value Model)."""
        rewards_t = torch.tensor(rewards, dtype=torch.float32)
        mean_r = rewards_t.mean()
        std_r = rewards_t.std() + 1e-8  # تفادي القسمة على صفر
        advantages = (rewards_t - mean_r) / std_r
        return advantages

    def compute_grpo_loss(self, log_probs_new, log_probs_old,
                           log_probs_ref, advantages):
        """
        حساب خسارة GRPO: PPO-style clipped objective + عقوبة KL صريحة
        (بخلاف PPO التي تدمج KL داخل المكافأة أحيانًا).
        """
        # نسبة الاحتمالات (policy ratio)
        ratio = torch.exp(log_probs_new - log_probs_old)

        # الهدف المقصوص (clipped surrogate objective) كما في PPO
        unclipped = ratio * advantages
        clipped = torch.clamp(
            ratio, 1 - self.config.clip_epsilon, 1 + self.config.clip_epsilon
        ) * advantages
        policy_loss = -torch.min(unclipped, clipped).mean()

        # عقوبة KL صريحة بين السياسة الحالية والمرجعية (منع الانحراف الكبير)
        kl_penalty = (
            torch.exp(log_probs_ref - log_probs_new)
            - (log_probs_ref - log_probs_new) - 1
        ).mean()  # تقدير KL غير المتحيّز (k3 estimator)

        total_loss = policy_loss + self.config.kl_coefficient * kl_penalty
        return total_loss, policy_loss.item(), kl_penalty.item()

    def train_step(self, prompt_ids, tokenizer, ground_truth_answer):
        """خطوة تدريب واحدة كاملة بمنهجية GRPO."""
        # 1. توليد مجموعة من الاستجابات
        responses = self.sample_group(prompt_ids, tokenizer)

        # 2. حساب المكافأة القابلة للتحقق لكل استجابة
        #    (مثال: 1.0 إذا كانت الإجابة الرياضية صحيحة، 0.0 إذا خاطئة
        #     + مكافأة تنسيق صغيرة إذا استُخدمت وسوم <think></think>)
        rewards = [
            self.reward_fn(resp, ground_truth_answer) for resp in responses
        ]

        # 3. حساب الـ Advantages النسبية (بدون Value Model!)
        advantages = self.compute_group_advantages(rewards)

        # 4. حساب log-probabilities من السياسة الحالية والمرجعية
        log_probs_new = self._compute_log_probs(self.policy_model, responses)
        log_probs_old = log_probs_new.detach()  # من نفس التوليد (on-policy تقريبًا)
        log_probs_ref = self._compute_log_probs(self.ref_model, responses)

        # 5. حساب الخسارة والتحديث
        loss, policy_loss, kl = self.compute_grpo_loss(
            log_probs_new, log_probs_old, log_probs_ref, advantages
        )
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_model.parameters(), 1.0)
        self.optimizer.step()

        return {"loss": loss.item(), "policy_loss": policy_loss,
                "kl": kl, "mean_reward": sum(rewards) / len(rewards)}

    def _compute_log_probs(self, model, responses):
        # دالة مساعدة مفاهيمية — تحسب مجموع log-probs للتسلسل بالكامل
        raise NotImplementedError("تُطبَّق بحسب واجهة النموذج المحدد")

def verifiable_math_reward(response_text: str, ground_truth: str) -> float:
    """
    دالة مكافأة قابلة للتحقق (Rule-Based Reward) — جوهر نجاح R1-Zero.
    بخلاف Reward Model مُدرَّب (قد يُخدَع بـ Reward Hacking)، هذه الدالة
    حتمية 100% ولا يمكن التلاعب بها.
    """
    import re
    # استخراج الإجابة النهائية من داخل \boxed{} أو بعد "الإجابة النهائية:"
    match = re.search(r"\\boxed\{([^}]+)\}", response_text)
    if not match:
        match = re.search(r"الإجابة النهائية[:\s]*([^\n]+)", response_text)
    if not match:
        return 0.0
    extracted = match.group(1).strip()

    # مكافأة الدقة
    correctness_reward = 1.0 if extracted == ground_truth.strip() else 0.0

    # مكافأة تنسيق إضافية صغيرة (تشجيع استخدام وسوم التفكير المنظّمة)
    format_reward = 0.1 if "<think>" in response_text and "</think>" in response_text else 0.0

    return correctness_reward + format_reward


> ⚠️ **خطأ شائع**: الاعتقاد بأن مكافآت Reward Model التقليدية (المدرَّبة على تفضيلات بشرية - الفصل 32) كافية لتدريب نماذج الاستدلال. في الواقع، **المكافآت القابلة للتحقق برمجيًا (Verifiable Rewards)** — كصحة معادلة رياضية، أو نجاح كود في اختبارات unit tests — هي ما مكّن DeepSeek-R1-Zero من التعلّم الذاتي بدون أي تدخل بشري في مرحلة SFT الأولى. هذا النهج يُعرف بـ **RLVR (Reinforcement Learning from Verifiable Rewards)**.

### 3.3 ظاهرة "لحظة الأها" (Aha Moment) والـ Backtracking

أحد أكثر الاكتشافات إثارة في تقرير DeepSeek-R1: أثناء تدريب RL المحض (R1-Zero)، طوّر النموذج **تلقائيًا** (دون أي توجيه بشري) سلوك التراجع عن مسار تفكير خاطئ واستئناف المحاولة من زاوية مختلفة، مُنتِجًا عبارات مثل:


```text
"...دعني أعيد التفكير في هذه الخطوة. ربما ارتكبتُ خطأً في الافتراض
الأول. لنجرّب نهجًا مختلفًا تمامًا..."
```


هذا السلوك لم يُدرَّب عليه صراحةً — بل **ظهر (Emergent)** كنتيجة طبيعية لتعظيم مكافأة الصحة عبر سلاسل تفكير طويلة. يُظهر هذا أن الاستدلال الطويل عبر RL يمكن أن يُفرز قدرات غير مُبرمجة مسبقًا، مشابهة لما ناقشناه في فصل Emergent Abilities (45).

### 3.4 Process Reward Model (PRM) مقابل Outcome Reward Model (ORM)

> ⚖️ **جدول مقارنة: ORM مقابل PRM**

| المعيار | Outcome Reward Model (ORM) | Process Reward Model (PRM) |
|---|---|---|
| ماذا يُقيّم | الإجابة النهائية فقط (صح/خطأ) | كل خطوة في سلسلة التفكير |
| تكلفة بيانات التدريب | رخيصة (تحقق تلقائي من الإجابة) | مكلفة جدًا (تحتاج تسمية كل خطوة يدويًا أو بـ MCTS) |
| دقة التوجيه | ضعيفة (لا تعرف أين الخطأ بالضبط) | دقيقة (تحدد الخطوة الدقيقة الخاطئة) |
| خطر Reward Hacking | أقل (يصعب خداع "الإجابة صحيحة أم لا") | أعلى (يمكن توليد خطوات "تبدو" صحيحة شكليًا) |
| الاستخدام الأمثل | RL على نطاق واسع (كما في R1) | إعادة الترتيب (Re-ranking) لعدة محاولات، أو توجيه بحث الشجرة |
| مثال استخدام | DeepSeek-R1 (RLVR) | OpenAI PRM800K، Let's Verify Step by Step |

### 3.5 استراتيجيات Test-Time Compute الأخرى

ليس التدريب عبر RL هو الطريق الوحيد لاستغلال "وقت الاستدلال". توجد أربع استراتيجيات رئيسية مكمِّلة:


In [ ]:
from enum import Enum

class TestTimeComputeStrategy(Enum):
    """الاستراتيجيات الأربع الرئيسية لاستغلال حوسبة وقت الاستدلال."""
    SEQUENTIAL_REVISION = "sequential_revision"      # النموذج يراجع إجابته تكراريًا
    PARALLEL_SAMPLING = "parallel_sampling"           # توليد N إجابات واختيار الأفضل
    TREE_SEARCH = "tree_search"                        # بحث شجري موجّه (ToT، MCTS)
    LEARNED_LONG_CHAIN = "learned_long_chain"          # RL على سلاسل تفكير طويلة (o1/R1)

def best_of_n_with_prm(model, prm_scorer, prompt: str, n: int = 32) -> str:
    """
    استراتيجية Best-of-N الموجّهة بـ PRM: وَلِّد N حلول، قيّم كل خطوة بـ PRM،
    اختر الحل ذا أعلى درجة تراكمية. أرخص من RL لكن أكثر تكلفة وقت الاستدلال
    من استدلال واحد مباشر.
    """
    candidates = [model.generate(prompt, temperature=0.8) for _ in range(n)]

    scored_candidates = []
    for candidate in candidates:
        steps = split_into_reasoning_steps(candidate)
        step_scores = [prm_scorer.score_step(prompt, steps[:i+1])
                       for i in range(len(steps))]
        # الدرجة التراكمية = أدنى درجة عبر الخطوات (أضعف حلقة في السلسلة)
        # أو حاصل الضرب — كلا الاستراتيجيتين مُستخدم في الأدبيات
        aggregate_score = min(step_scores) if step_scores else 0.0
        scored_candidates.append((candidate, aggregate_score))

    best_candidate = max(scored_candidates, key=lambda x: x[1])
    return best_candidate[0]

def split_into_reasoning_steps(text: str) -> list:
    """تقسيم نص التفكير إلى خطوات منفصلة (بناءً على فواصل الأسطر/الترقيم)."""
    import re
    steps = re.split(r'\n(?=\d+[\.\)]|الخطوة)', text)
    return [s.strip() for s in steps if s.strip()]


> 📐 **منحنى التحجيم: Test-Time Compute مقابل Train-Time Compute**

ورقة "Scaling LLM Test-Time Compute Optimally" (Snell et al., 2024) تُظهر نتيجة محورية:


```text
لمهمة بصعوبة متوسطة:
  زيادة Test-Time Compute (عبر Best-of-N أو PRM-guided search)
  يمكن أن تُعادل أداء نموذج أكبر بـ 14× من ناحية Train-Time Compute
  (لكن فقط ضمن حدود صعوبة السؤال — المهام الصعبة جدًا لا تزال
   تحتاج نموذجًا أساسيًا أقوى، لا مجرد "وقت تفكير" أطول)
```


## 4. مثال محلول خطوة بخطوة


```text
السؤال: "ما ناتج 17 × 24 - 15²؟"

<think>
لنحسب هذا خطوة بخطوة.
أولًا: 17 × 24 = ؟
17 × 24 = 17 × 20 + 17 × 4 = 340 + 68 = 408

ثانيًا: 15² = 225

إذن الناتج = 408 - 225 = 183

دعني أتحقق من الحساب الأول مرة أخرى...
17 × 24: يمكن أيضًا حسابها كـ (20-3) × 24 = 480 - 72 = 408 ✓ (نفس النتيجة، جيد)

15² = 15 × 15 = 225 ✓ (صحيح)

408 - 225 = 183 ✓
</think>

الإجابة النهائية: \boxed{183}
```


لاحظ أن النموذج **تحقق من نفسه مرتين** بطريقتين حسابيتين مختلفتين قبل الجزم بالإجابة — هذا هو جوهر القيمة المضافة لنماذج الاستدلال: استهلاك رموز إضافية مقابل موثوقية أعلى.


## 5. نصائح وحيل

- عند استخدام نماذج استدلال (o1/o3/DeepSeek-R1) عبر API، **لا تستخدم Few-Shot Prompting التقليدي ولا تطلب "فكّر خطوة بخطوة"** — هذه النماذج مُدرَّبة بالفعل على التفكير الداخلي، وإضافة أمثلة يدوية قد تُربك نمط تفكيرها الطبيعي وتزيد التكلفة دون فائدة.
- في OpenAI o1/o3، استخدم معامل `reasoning_effort` (low/medium/high) للتحكم في عمق التفكير مقابل الكلفة والزمن.
- لمهام الترميز والرياضيات ذات إجابة قابلة للتحقق تلقائيًا، نماذج الاستدلال تتفوق بفارق كبير؛ لمهام المحادثة العامة أو الكتابة الإبداعية، الفائدة أقل وضوحًا والتكلفة أعلى بلا مبرر.
- عند بناء نظام إنتاجي، استخدم **توجيهًا ذكيًا (Model Routing)** كما في الفصل 59: وجّه الأسئلة المعقدة فقط (رياضيات، منطق متعدد الخطوات، تصحيح كود) إلى نموذج الاستدلال، والباقي إلى نموذج أسرع وأرخص.


## 6. أخطاء شائعة ومصائد

- **الخلط بين "طول الاستجابة" و"جودة التفكير"**: نموذج قد يولّد تفكيرًا طويلًا جدًا دون أن يتحسّن أداؤه الفعلي (ظاهرة "Overthinking")؛ قياس الأداء الفعلي (دقة الإجابة) أهم من عدّ الرموز.
- تجاهل أن **تكلفة الاستدلال** لنماذج الاستدلال أعلى بأضعاف (أحيانًا 10-50×) من نموذج تقليدي بنفس الحجم؛ يجب تضمين هذا في حسابات التكلفة (LLMCostCalculator - الفصل 59).
- افتراض أن RLVR يعمل مع أي نوع مهمة — في الواقع تحتاج **دالة تحقق تلقائي موثوقة** (كود يعمل، معادلة رياضية صحيحة)؛ المهام ذات الإجابات الذاتية (مثل: "اكتب قصيدة جميلة") لا تمتلك إشارة تحقق واضحة، لذا RLVR أقل فعالية هناك.


## 7. جدول مقارنة

| النموذج | الجهة | منهجية التدريب | الشفافية | الأداء على AIME 2024 |
|---|---|---|---|---|
| OpenAI o1 | OpenAI | RL + سلاسل تفكير مخفية | مغلق تمامًا | ~74% |
| OpenAI o3 | OpenAI | RL موسّع + test-time compute أعلى | مغلق تمامًا | ~96% (تقديري، إعداد عالي الكلفة) |
| DeepSeek-R1 | DeepSeek | SFT Cold-Start + GRPO (RLVR) | أوزان مفتوحة بالكامل + ورقة تفصيلية | ~79.8% |
| DeepSeek-R1-Zero | DeepSeek | RL محض بدون SFT | أوزان مفتوحة | ~71% (لكن لغة مختلطة أحيانًا) |
| QwQ (Alibaba) | Alibaba | RL على سلاسل تفكير | أوزان مفتوحة | تنافسي، أقل من R1 |
| Gemini 2.0 Flash Thinking | Google | RL + test-time compute | مغلق جزئيًا (سلاسل تفكير مرئية) | تنافسي |


## 8. ملخص أفضل الممارسات

- ✅ استخدم المكافآت القابلة للتحقق (RLVR) كلما أمكن بدل Reward Models ذاتية التدريب عند استهداف مهام منطقية/رياضية/برمجية.
- ✅ افصل بين "نموذج يفكّر طويلًا بفطرته المُدرَّبة" و"نموذج يُطلب منه التفكير عبر Prompt" — الأول أقوى بكثير وأكثر موثوقية.
- ✅ استخدم Model Routing لتوجيه الأسئلة الصعبة فقط لنماذج الاستدلال المكلفة.
- ✅ قس "Overthinking" ودقق أن طول التفكير يتناسب فعلًا مع تحسّن الدقة قبل اعتماد إعداد `reasoning_effort` عالٍ في الإنتاج.
- ✅ إن كنت تبني PRM، استثمر في بيانات تسمية الخطوات الفردية (قد تستخدم MCTS لتوليدها تلقائيًا كما في ورقة Math-Shepherd).


## 9. أسئلة مقابلات

**سؤال 1:** ما الفرق الجوهري بين Test-Time Scaling وTrain-Time Scaling، ولماذا يُعتبر الأول اكتشافًا مهمًا؟

> **إجابة نموذجية:** Train-Time Scaling (قوانين Chinchilla) يُحسِّن الأداء عبر زيادة حجم النموذج (N) أو حجم بيانات التدريب (D) — وهذا يتطلب إعادة تدريب مكلفة جدًا لكل تحسين. Test-Time Scaling يُحسِّن الأداء عبر زيادة الحوسبة المُستهلكة **وقت الاستدلال فقط** (توليد سلاسل تفكير أطول، أو توليد عدة محاولات واختيار الأفضل) **دون تغيير أوزان النموذج**. الأهمية: (1) يفتح مسارًا جديدًا للتحسين مستقلًا عن حدود التدريب، (2) يسمح بالمرونة — يمكن للمستخدم "دفع" المزيد للحصول على جودة أعلى فقط عند الحاجة (سؤال صعب)، بخلاف التدريب الذي يُثبّت القدرة بشكل دائم بغض النظر عن صعوبة كل سؤال على حدة.

**سؤال 2:** اشرح لماذا استغنت GRPO عن Value Model (Critic) الذي كان ضروريًا في PPO التقليدي، وما الثمن المُقابل لذلك؟

> **إجابة نموذجية:** في PPO، يحتاج النموذج لتقدير baseline (القيمة المتوقعة للحالة) عبر Value Model منفصل مُدرَّب بالتوازي لحساب الـ Advantage = Reward - Value؛ هذا يضاعف تقريبًا استهلاك الذاكرة وتعقيد التدريب. GRPO تستبدل هذا بحساب متوسط وانحراف معياري مكافآت **مجموعة من الاستجابات لنفس السؤال** واستخدامها كـ baseline إحصائي (Advantage = (reward - mean)/std). الثمن: يجب توليد عدة عينات (مثلاً 16) لكل سؤال في كل خطوة تدريب، مما يزيد تكلفة الاستدلال أثناء التدريب (لكن هذا أرخص بكثير من تدريب وصيانة Value Model منفصل بحجم مشابه للـ Policy Model).

**سؤال 3:** ما هي "لحظة الأها" (Aha Moment) في تقرير DeepSeek-R1، ولماذا تُعتبر دليلًا على Emergent Behavior؟

> **إجابة نموذجية:** هي لحظة ظهور سلوك التراجع التلقائي (Backtracking) في سلاسل تفكير R1-Zero — حيث يتوقف النموذج في منتصف حل ويقول عبارات مثل "دعني أعيد التفكير"، رغم أن هذا السلوك لم يُدرَّج صراحةً في دالة المكافأة ولا في أي بيانات SFT (لأن R1-Zero دُرِّب بـ RL محض بدون SFT أولي). ظهور هذا السلوك تلقائيًا كاستراتيجية ناجحة لتعظيم مكافأة الصحة يُعتبر دليلًا على Emergent Behavior، مشابهًا لمفهوم القدرات الناشئة الذي ناقشناه في الفصل 45، لكنه هنا ناتج عن RL وليس عن مجرد زيادة حجم النموذج.


## 10. تمرين عملي

نفّذ دالة `reward_fn` قابلة للتحقق لمهمة توليد كود بايثون: تأخذ الكود المُولَّد ومجموعة من test cases، وتُشغّل الكود فعليًا (في بيئة sandbox آمنة)، وتُعيد 1.0 إذا اجتاز كل الاختبارات، أو نسبة الاختبارات الناجحة كمكافأة جزئية. ثم استخدم `GRPOTrainer` (من القسم 3.2) لمحاكاة خطوة تدريب واحدة باستخدام هذه الدالة على سؤال برمجي بسيط (مثل: "اكتب دالة لحساب العدد الأولي التالي").


## 11. مراجع للتعمّق

- DeepSeek-AI (2025): *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning*
- OpenAI (2024): *Learning to Reason with LLMs* (تقرير o1)
- Snell et al. (2024): *Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Parameters*
- Lightman et al. (2023): *Let's Verify Step by Step* (PRM800K)
- Wang et al. (2024): *Math-Shepherd: Verify and Reinforce LLMs Step-by-step without Human Annotations*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الاتجاهات المستقبلية / الفصل 60: نماذج الاستدلال وTest-Time Compute.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
